In [1]:
import tifffile
import spatialdata as sd 
from napari_spatialdata import Interactive
from spatialdata.models import Image2DModel
from dask_image.imread import imread
import numpy as np
from glob import glob

In [1]:
#!/usr/bin/env python

import tifffile

# Variable 'LEVEL' determines the level to extract. It ranges from 0 (highest
# resolution) to 6 (lowest resolution) for morphology.ome.tif
LEVEL = 0

with tifffile.TiffFile('Version 2//Sample 4H//morphology_focus.ome.tif') as tif:
    image = tif.series[0].levels[LEVEL].asarray()

tifffile.imwrite(
    'Version 2 DAPI//Version_2_Sample_4H_level_'+str(LEVEL)+'_morphology_focus.ome.tif',
    image,
    photometric='minisblack',
    dtype='uint16',
    tile=(1024, 1024),
    compression='JPEG_2000_LOSSY',
    metadata={'axes': 'YX'},
)

In [3]:
with tifffile.TiffFile('Version 1//Sample 1D//morphology_focus.ome.tif') as tif:
    for tag in tif.pages[0].tags.values():
        if tag.name == "ImageDescription":
            print(tag.name+":", tag.value)

ImageDescription: <OME xmlns="http://www.openmicroscopy.org/Schemas/OME/2016-06" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" Creator="tifffile.py 2022.10.10" UUID="urn:uuid:f3e91288-3620-11ee-8082-3cecefcca217" xsi:schemaLocation="http://www.openmicroscopy.org/Schemas/OME/2016-06 http://www.openmicroscopy.org/Schemas/OME/2016-06/ome.xsd">
    <Plate ID="Plate:1" WellOriginX="-0.0" WellOriginXUnit="µm" WellOriginY="-0.0" WellOriginYUnit="µm" />
    <Instrument ID="Instrument:1">
        <Microscope Manufacturer="10x Genomics" Model="Xenium" />
    </Instrument>
    <Image ID="Image:0" Name="Image0">
        <InstrumentRef ID="Instrument:1" />
        <Pixels DimensionOrder="XYZCT" ID="Pixels:0" SizeC="1" SizeT="1" SizeX="28466" SizeY="27254" SizeZ="1" Type="uint16" PhysicalSizeX="0.2125" PhysicalSizeY="0.2125">
            <Channel ID="Channel:0:0" Name="DAPI" SamplesPerPixel="1" />
            <TiffData PlaneCount="1" />
        </Pixels>
    </Image>
</OME>


In [ ]:
image.shape

(27254, 28466)

In [4]:
tifffile.imwrite(
    'level_'+str(LEVEL)+'_morphology_focus.ome.tif',
    image,
    photometric='minisblack',
    dtype='uint16',
    tile=(1024, 1024),
    compression='JPEG_2000_LOSSY',
    metadata={'axes': 'YX'},
)

In [6]:
Xenium = sd.read_zarr('Version 1 Zarr//Sample1H.zarr')

C:\Users\laure\AppData\Roaming\Python\Python39\site-packages\anndata\_core\anndata.py:117: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


In [14]:
image = imread("level_2_morphology_focus.ome.tif")
Xenium.images["level_2_morphology_focus.ome.tif"] = Image2DModel.parse(image, dims=("c","x","y"), scale_factors=(2,2,2))

INFO     Transposing `data` of type: <class 'dask.array.core.Array'> to ('c', 'y', 'x').                           


In [12]:
image

dask.array<_map_read_frame, shape=(1, 4275, 7116), dtype=uint16, chunksize=(1, 4275, 7116), chunktype=numpy.ndarray>

In [15]:
napari = Interactive(Xenium)

In [2]:
fnames = glob('*morphology_focus.ome_seg.npy')

for f in fnames:
    dat = np.load(f, allow_pickle=True).item()

    for key in dat.keys():
        dtype = type(dat[key])
        print("Key:", key)
        print("Dtype:", dtype)

        if isinstance(dat[key], np.ndarray):
            print("Dimension:", dat[key].shape)
        elif isinstance(dat[key], list):
            print("Length:", len(dat[key]))

        # empty line between keys
        print()

Key: outlines
Dtype: <class 'numpy.ndarray'>
Dimension: (4275, 7116)

Key: colors
Dtype: <class 'numpy.ndarray'>
Dimension: (4089, 3)

Key: masks
Dtype: <class 'numpy.ndarray'>
Dimension: (4275, 7116)

Key: chan_choose
Dtype: <class 'list'>
Length: 2

Key: filename
Dtype: <class 'str'>

Key: flows
Dtype: <class 'list'>
Length: 2

Key: ismanual
Dtype: <class 'numpy.ndarray'>
Dimension: (4089,)

Key: manual_changes
Dtype: <class 'list'>
Length: 0

Key: model_path
Dtype: <class 'str'>

Key: flow_threshold
Dtype: <class 'float'>

Key: cellprob_threshold
Dtype: <class 'float'>

Key: normalize_params
Dtype: <class 'dict'>

Key: restore
Dtype: <class 'NoneType'>

Key: ratio
Dtype: <class 'float'>

Key: diameter
Dtype: <class 'float'>

